Limpieza y Cargue de Datos:
Censo 2018 Comunas de Manizales.
En este cuaderno realizaremos el proceso de obtención, transformación y depuración de datos utilizando la librería Pandas.

In [1]:
# Importamos las librerías necesarias para la manipulación de datos 
import pandas as pd 
import numpy as np 
import os 
print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


Paso 1: 
Cargar los datos desde los archivos de origen.
Leemos el archivo CSV con la información del Censo de Manizales y los archivos de ventas.

In [2]:
# Cargamos el archivo del Censo de Manizales indicando que lea todo como texto inicialmente 
df_censo = pd.read_csv('Censo_2018_Comunas_de_Manizales.csv', dtype=str) 

# Cargamos los archivos de ventas privadas 
df_ventas_centro = pd.read_csv('ventas_centro.csv') 
df_ventas_sur = pd.read_csv('ventas_sur.csv') 

# Mostramos las dimensiones (filas y columnas) del dataset del censo 
print(f"Dimensiones del Censo de Manizales: {df_censo.shape[0]} filas x {df_censo.shape[1]} columnas")

Dimensiones del Censo de Manizales: 184 filas x 15 columnas


Paso 2:
Aplanar la tabla (Transformar de formato ancho a formato largo).
El archivo original tiene 13 columnas de territorio (Comunas 01 a 12 y Zona Rural). 
Usamos pd.melt() para aplanarlas a una sola columna vertical.

In [3]:
# Despivotamos las columnas de territorio usando pd.melt() 
df_censo_todo = pd.melt(df_censo, id_vars=['Variable', 'Categoría'], var_name='Territorio', value_name='Valor_Texto') 

print(f"Nuevas dimensiones tras aplanar: {df_censo_todo.shape[0]} filas x {df_censo_todo.shape[1]} columnas")

Nuevas dimensiones tras aplanar: 2392 filas x 4 columnas


Paso 3:
Limpiar el formato numérico y verificar nulos.
Convertimos los valores de texto (que tienen puntos de miles) a números flotantes y verificamos si hay datos faltantes.

In [5]:
# Eliminamos los puntos de miles y convertimos a número flotante 
df_censo_todo['Valor_Numerico'] = (df_censo_todo['Valor_Texto'].str.replace('.', '', regex=False) .astype(float)) 

# Revisamos la cantidad de datos nulos por columna 
print("=== Conteo de Valores Nulos ===") 
print(df_censo_todo.isnull().sum())

=== Conteo de Valores Nulos ===
Variable          0
Categoría         0
Territorio        0
Valor_Texto       0
Valor_Numerico    0
dtype: int64


Paso 4: 
Unificar las fuentes de ventas operacionales. 
Unimos las tablas de ventas del centro y del sur en un único DataFrame usando pd.concat().

In [6]:
# Concatenamos las tablas de ventas verticalmente 
df_ventas_unificadas = pd.concat([df_ventas_centro, df_ventas_sur], ignore_index=True) 

print(f"Total de registros de ventas unificados: {len(df_ventas_unificadas)}")

Total de registros de ventas unificados: 20


Paso 5: 
Filtrar un indicador clave de Manizales (Internet por Comuna).
Filtramos las viviendas que cuentan con servicio de internet en cada comuna para analizar la cobertura territorial.

In [9]:
# Filtramos la variable de internet y la categoría 'Si' 
filtro_internet = (df_censo_todo['Variable'].str.contains('internet', case=False, na=False)) & (df_censo_todo['Categoría'] == 'Si') 

df_internet = df_censo_todo[filtro_internet][['Territorio', 'Valor_Numerico']].copy() 
df_internet.columns = ['Territorio', 'Viviendas_Con_Internet']

print("=== Viviendas con Internet por Comuna en Manizales ===") 
print(df_internet)

=== Viviendas con Internet por Comuna en Manizales ===
      Territorio  Viviendas_Con_Internet
155    Comuna 01                 27500.0
339    Comuna 02                  4019.0
523    Comuna 03                 15046.0
707    Comuna 04                 18264.0
891    Comuna 05                 31147.0
1075   Comuna 06                 25352.0
1259   Comuna 07                 18987.0
1443   Comuna 08                 24301.0
1627   Comuna 09                 23477.0
1811   Comuna 10                 28473.0
1995   Comuna 11                 21691.0
2179   Comuna 12                 23736.0
2363  Zona Rural                  4567.0


Paso 6: 
Exportar el dataset limpio a la zona de Staging.
Guardamos el resultado procesado en un archivo CSV listo para su análisis en herramientas de BI o modelos de datos.

In [10]:
# Exportamos el DataFrame limpio a un archivo CSV sin incluir el índice 
df_censo_todo.to_csv('staging_censo_ventas_manizales_limpio.csv', index=False)

print("¡Proceso completado! El dataset ha sido exportado exitosamente.")

¡Proceso completado! El dataset ha sido exportado exitosamente.
